# Spider Expanded — воспроизводимая сборка на Kaggle


In [ ]:
from pathlib import Path
import os, subprocess, sys
WORK = Path("/kaggle/working")
REPO = WORK / "BaseModel-vs-SFT"
PINNED_REF = "e739df49b1b87f200d7d95428bb09b0a97686f3d"
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/Pavel-V418/BaseModel-vs-SFT.git", str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "origin", "dataset/expanded-v1-publication"], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", PINNED_REF], check=True)
print("Pinned repository code:", PINNED_REF)


## 1. Загрузить код проекта


In [ ]:
import importlib.util
if importlib.util.find_spec("gdown") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
import gdown, urllib.request, hashlib
SPIDER_ZIP = WORK / "spider_data.zip"
EXPECTED_ARCHIVE_SHA256 = "00636695dabed6b5f4b8328a16b13e069a2f16591d5efcce57660669c85b121b"
def digest(path):
    hash_obj = hashlib.sha256()
    with open(path, "rb") as f:
        for buf in iter(lambda: f.read(1024 * 1024), b""):
            hash_obj.update(buf)
    return hash_obj.hexdigest()
if SPIDER_ZIP.exists() and digest(SPIDER_ZIP) != EXPECTED_ARCHIVE_SHA256:
    SPIDER_ZIP.unlink()
if not SPIDER_ZIP.exists():
    try:
        gdown.download(id="1403EGqzIDoHMdQF4c9Bkyl7dZLZ5Wt6J", output=str(SPIDER_ZIP), quiet=False)
        if not SPIDER_ZIP.exists() or digest(SPIDER_ZIP) != EXPECTED_ARCHIVE_SHA256:
            raise ValueError("Downloaded Google Drive archive hash does not match")
    except Exception as exc:
        print("Google Drive failed; trying known Hugging Face mirror:", exc)
        if SPIDER_ZIP.exists():
            SPIDER_ZIP.unlink()
        url = "https://huggingface.co/datasets/HAL-9001/spider-databases/resolve/main/spider_data.zip"
        urllib.request.urlretrieve(url, SPIDER_ZIP)
if digest(SPIDER_ZIP) != EXPECTED_ARCHIVE_SHA256:
    raise RuntimeError("Unexpected Spider ZIP SHA-256. Stop; do not build from mismatched data.")
print("Source Spider ZIP verified:", digest(SPIDER_ZIP))


## 2. Скачать официальный Spider и проверить SHA-256


In [ ]:
import zipfile, shutil
SPIDER_DIR = WORK / "spider_expanded_build" / "spider"
SPIDER_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(SPIDER_ZIP) as archive:
    for item in archive.infolist():
        if not item.filename.startswith("spider_data/") or item.is_dir():
            continue
        relative = Path(item.filename).relative_to("spider_data")
        if ".." in relative.parts:
            raise ValueError("Unsafe archive path")
        target = SPIDER_DIR / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        with archive.open(item) as source, target.open("wb") as dest:
            shutil.copyfileobj(source, dest)
for name in ("train_spider.json", "train_others.json", "dev.json", "tables.json"):
    if not (SPIDER_DIR / name).is_file():
        raise FileNotFoundError(SPIDER_DIR / name)
print("Official Spider extracted to:", SPIDER_DIR)


## 3. Распаковать исходные базы и JSON


In [ ]:
OUT = WORK / "spider_expanded_build" / "data"
cmd = [sys.executable, "-m", "src.data", "--spider_dir", str(SPIDER_DIR), "--out_dir", str(OUT),
       "--val_size", "300", "--seed", "42", "--include_others", "--tokenizer", "", "--show", "0"]
print("Build command:", " ".join(cmd))
subprocess.run(cmd, cwd=REPO, check=True)
import json
stats = json.loads((OUT / "stats.json").read_text(encoding="utf-8"))
print({key: stats["splits"][key]["n_examples"] for key in ("train", "val", "test")})
print("Dropped:", stats["n_dropped_total"])


## 4. Сформировать train/val/test через src.data


In [ ]:
RELEASE = WORK / "spider_expanded_release"
cmd = [sys.executable, str(REPO / "scripts" / "audit_expanded_dataset.py"),
       "--data-dir", str(OUT), "--out-dir", str(RELEASE), "--archive"]
try:
    subprocess.run(cmd, check=True)
except subprocess.CalledProcessError:
    print("AUDIT FAILED. See printed discrepancy; do not label the regenerated data historical 8222/300.")
    raise
print("Prepared candidate ZIP:", RELEASE / "spider_expanded_v1_private.zip")
print("Manifest:", RELEASE / "expanded_v1_manifest.json")


## 5. Проверить и упаковать


In [ ]:
from IPython.display import display, FileLink
display(FileLink(str(RELEASE / "spider_expanded_v1_private.zip")))
display(FileLink(str(RELEASE / "expanded_v1_manifest.json")))
print("Save Version -> Save & Run All to preserve outputs after Kaggle session.")
